# MNIST LDA & QDA Classifier
Trains Linear Discriminant Analysis (LDA) and Quadratic Discriminant Analysis (QDA) classifiers on MNIST digits 0, 1, 2 (100 samples/class). Visualizes class structure with t-SNE and discriminant scores.


In [ ]:
import numpy as np
import struct
from os.path import join
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist   # avoids local download — loads from Keras cache


## Load Data
Using `keras.datasets.mnist` — no manual download required. Data is cached in `~/.keras/datasets/`.

In [ ]:
(x_train_full, y_train_full), (x_test_full, y_test_full) = mnist.load_data()

# Keep only digits 0, 1, 2
train_mask = np.isin(y_train_full, [0, 1, 2])
test_mask  = np.isin(y_test_full,  [0, 1, 2])

x_train, y_train = x_train_full[train_mask], y_train_full[train_mask]
x_test,  y_test  = x_test_full[test_mask],   y_test_full[test_mask]

print(f"Train samples: {len(x_train)} | Test samples: {len(x_test)}")


## Sample 100 Images per Class

In [ ]:
np.random.seed(42)

def sample_100(x, y, cls):
    xc, yc = x[y == cls], y[y == cls]
    idx = np.random.choice(len(xc), 100, replace=False)
    return xc[idx], yc[idx]

x_train_0, y_train_0 = sample_100(x_train, y_train, 0)
x_train_1, y_train_1 = sample_100(x_train, y_train, 1)
x_train_2, y_train_2 = sample_100(x_train, y_train, 2)

x_test_0, y_test_0 = sample_100(x_test, y_test, 0)
x_test_1, y_test_1 = sample_100(x_test, y_test, 1)
x_test_2, y_test_2 = sample_100(x_test, y_test, 2)

x_test_all  = np.concatenate([x_test_0,  x_test_1,  x_test_2])
y_test_all  = np.array([0]*100 + [1]*100 + [2]*100)


## Preprocess
Normalise to [0, 1] and flatten each 28×28 image to a 784-d column vector.

In [ ]:
def preprocess(x):
    """Normalise and return as (784, N) matrix.""\"""
    x = x.astype(np.float32) / 255.0
    return x.reshape(x.shape[0], -1).T

X0, X1, X2 = preprocess(x_train_0), preprocess(x_train_1), preprocess(x_train_2)
X_test      = preprocess(x_test_all)

print("X0 shape:", X0.shape, "| X_test shape:", X_test.shape)


## Class Means & Covariances

In [ ]:
mu0 = np.mean(X0, axis=1, keepdims=True)
mu1 = np.mean(X1, axis=1, keepdims=True)
mu2 = np.mean(X2, axis=1, keepdims=True)

def cov(X, mu):
    Xc = X - mu
    return (Xc @ Xc.T) / (X.shape[1] - 1)

eps = 1e-2   # regularisation to ensure invertibility

Sigma0 = cov(X0, mu0) + eps * np.eye(784)
Sigma1 = cov(X1, mu1) + eps * np.eye(784)
Sigma2 = cov(X2, mu2) + eps * np.eye(784)

P0 = P1 = P2 = 1/3   # equal priors


## Quadratic Discriminant Analysis (QDA)
Each class gets its own covariance matrix. The discriminant function is:

$$g_k(x) = -\frac{1}{2}(x-\mu_k)^\top \Sigma_k^{-1}(x-\mu_k) - \frac{1}{2}\ln|\Sigma_k|$$

`slogdet` is used for numerical stability with high-dimensional covariance matrices.


In [ ]:
S0_inv = np.linalg.inv(Sigma0)
S1_inv = np.linalg.inv(Sigma1)
S2_inv = np.linalg.inv(Sigma2)

_, logdet0 = np.linalg.slogdet(Sigma0)
_, logdet1 = np.linalg.slogdet(Sigma1)
_, logdet2 = np.linalg.slogdet(Sigma2)

def g_QDA(x, mu, S_inv, logdetS):
    d = x - mu
    return -0.5 * (d.T @ S_inv @ d) - 0.5 * logdetS

def predict_QDA(x):
    x = x.reshape(-1, 1)
    scores = [
        g_QDA(x, mu0, S0_inv, logdet0),
        g_QDA(x, mu1, S1_inv, logdet1),
        g_QDA(x, mu2, S2_inv, logdet2),
    ]
    return np.argmax(scores)

pred_QDA = np.array([predict_QDA(X_test[:, i]) for i in range(X_test.shape[1])])
qda_acc  = np.mean(pred_QDA == y_test_all) * 100
print(f"QDA Accuracy: {qda_acc:.2f}%")


## Linear Discriminant Analysis (LDA)
All classes share a pooled covariance matrix. The discriminant function simplifies to:

$$g_k(x) = x^\top \Sigma^{-1}\mu_k - \frac{1}{2}\mu_k^\top \Sigma^{-1}\mu_k$$


In [ ]:
Sigma_pooled = (Sigma0 + Sigma1 + Sigma2) / 3 + eps * np.eye(784)
S_inv        = np.linalg.inv(Sigma_pooled)

def g_LDA(x, mu, S_inv):
    return x.T @ S_inv @ mu - 0.5 * mu.T @ S_inv @ mu

def predict_LDA(x):
    x = x.reshape(-1, 1)
    scores = [
        g_LDA(x, mu0, S_inv),
        g_LDA(x, mu1, S_inv),
        g_LDA(x, mu2, S_inv),
    ]
    return np.argmax(scores)

pred_LDA = np.array([predict_LDA(X_test[:, i]) for i in range(X_test.shape[1])])
lda_acc  = np.mean(pred_LDA == y_test_all) * 100
print(f"LDA Accuracy: {lda_acc:.2f}%")


## Accuracy Summary

In [ ]:
print(f"{'Model':<10} {'Accuracy':>10}")
print("-" * 22)
print(f"{'QDA':<10} {qda_acc:>9.2f}%")
print(f"{'LDA':<10} {lda_acc:>9.2f}%")


## t-SNE Visualisation
t-SNE projects the 784-d pixel space to 2D for visualisation. Note: t-SNE axes have no absolute meaning — only relative distances matter.


In [ ]:
X_train_all  = np.hstack((X0, X1, X2)).T
y_train_viz  = np.array([0]*100 + [1]*100 + [2]*100)

print("Running t-SNE on training set…")
X_train_tsne = TSNE(n_components=2, random_state=42).fit_transform(X_train_all)

print("Running t-SNE on test set…")
X_test_tsne  = TSNE(n_components=2, random_state=42).fit_transform(X_test.T)

print("Done.")


## Discriminant Scores for a Single Test Sample

In [ ]:
sample_idx  = 0
x_sample    = X_test[:, sample_idx].reshape(-1, 1)
true_label  = y_test_all[sample_idx]

scores_LDA = [
    g_LDA(x_sample, mu0, S_inv).item(),
    g_LDA(x_sample, mu1, S_inv).item(),
    g_LDA(x_sample, mu2, S_inv).item(),
]
scores_QDA = [
    g_QDA(x_sample, mu0, S0_inv, logdet0).item(),
    g_QDA(x_sample, mu1, S1_inv, logdet1).item(),
    g_QDA(x_sample, mu2, S2_inv, logdet2).item(),
]

print(f"True label : {true_label}")
print(f"LDA scores : {[f'{s:.2f}' for s in scores_LDA]}  → predicted {np.argmax(scores_LDA)}")
print(f"QDA scores : {[f'{s:.2f}' for s in scores_QDA]}  → predicted {np.argmax(scores_QDA)}")


## Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
colors = ['tab:blue', 'tab:orange', 'tab:green']

# Train t-SNE
for cls in [0, 1, 2]:
    mask = y_train_viz == cls
    axes[0].scatter(X_train_tsne[mask, 0], X_train_tsne[mask, 1],
                    label=f"Digit {cls}", alpha=0.7, s=20, color=colors[cls])
axes[0].set_title("Train Set — t-SNE Projection")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Test t-SNE
for cls in [0, 1, 2]:
    mask = y_test_all == cls
    axes[1].scatter(X_test_tsne[mask, 0], X_test_tsne[mask, 1],
                    label=f"Digit {cls}", alpha=0.7, s=20, color=colors[cls])
axes[1].set_title("Test Set — t-SNE Projection")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# LDA scores
bars = axes[2].bar(['Class 0', 'Class 1', 'Class 2'], scores_LDA, color=colors)
axes[2].set_title(f"LDA Scores (sample idx={sample_idx}, true={true_label})")
axes[2].set_ylabel("Score"); axes[2].grid(axis='y', alpha=0.3)

# QDA scores
axes[3].bar(['Class 0', 'Class 1', 'Class 2'], scores_QDA, color=colors)
axes[3].set_title(f"QDA Scores (sample idx={sample_idx}, true={true_label})")
axes[3].set_ylabel("Score"); axes[3].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("visualization.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → visualization.png")
